In [1]:
# Install dependencies
!pip install -U kubeflow torch --quiet

In [9]:
import time
from datetime import datetime
import kubeflow.trainer

In [10]:
config = kubeflow.trainer.KubernetesBackendConfig()
trainer = kubeflow.trainer.TrainerClient(backend_config=config)

In [4]:
# Set your distributed environment configuration here
NUM_NODES = 2
RESOURCES_PER_NODE = {
    "nvidia.com/gpu": 1,  # GPUs per node
    "cpu": "2",           # CPUs per node
    "memory": "8Gi"       # Memory in GiB per node
}


In [5]:
def get_torch_dist():
    import os
    import torch
    import torch.distributed as dist

    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    dist.init_process_group(backend)
    print("PyTorch Distributed Environment")
    print(f"Using device: {device}")
    print(f"WORLD_SIZE: {dist.get_world_size()}")
    print(f"RANK: {dist.get_rank()}")
    print(f"LOCAL_RANK: {os.environ['LOCAL_RANK']}")
    dist.destroy_process_group()


In [6]:
job_id = trainer.train(
    runtime=trainer.get_runtime("torch-distributed"),
    trainer=kubeflow.trainer.CustomTrainer(
        func=get_torch_dist,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
    ),
)

In [7]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")



Job ID: h0a1c8b098ed
Job Status: Created
Creation Time: 2026-01-14 15:54:11+00:00

Job details: TrainJob(name='h0a1c8b098ed', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=2, creation_timestamp=datetime.datetime(2026, 1, 14, 15, 54, 11, tzinfo=TzInfo(0)), status='Created')


In [11]:
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=False))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


Waiting for job logs...
[15:54:52] Waiting... (1s)
[15:54:53] Waiting... (2s)
[15:54:54] Waiting... (3s)
[15:54:55] Waiting... (4s)
[15:54:56] Waiting... (5s)
[15:54:57] Waiting... (6s)
[15:54:58] Waiting... (7s)
[15:54:59] Waiting... (8s)
[15:55:00] Waiting... (9s)
[15:55:01] Waiting... (10s)
[15:55:02] Waiting... (11s)
[15:55:03] Waiting... (12s)
Logs received after 12 seconds:
  [W114 15:55:03.289124830 socket.cpp:755] [c10d] The IPv6 network addresses of (h0a1c8b098ed-node-0-0.h0a1c8b098ed, 29500) cannot be retrieved (gai error: -3 - Temporary failure in name resolution).


In [ ]:
for logline in trainer.get_job_logs(job_id, follow=True):
    print(logline)

[W114 15:55:03.289124830 socket.cpp:755] [c10d] The IPv6 network addresses of (h0a1c8b098ed-node-0-0.h0a1c8b098ed, 29500) cannot be retrieved (gai error: -3 - Temporary failure in name resolution).
[W114 15:55:16.217093024 socket.cpp:755] [c10d] The IPv6 network addresses of (h0a1c8b098ed-node-0-0.h0a1c8b098ed, 29500) cannot be retrieved (gai error: -3 - Temporary failure in name resolution).
[W114 15:55:30.465086634 socket.cpp:755] [c10d] The IPv6 network addresses of (h0a1c8b098ed-node-0-0.h0a1c8b098ed, 29500) cannot be retrieved (gai error: -3 - Temporary failure in name resolution).
[W114 15:55:43.353106302 socket.cpp:755] [c10d] The IPv6 network addresses of (h0a1c8b098ed-node-0-0.h0a1c8b098ed, 29500) cannot be retrieved (gai error: -3 - Temporary failure in name resolution).
[W114 15:55:57.433092124 socket.cpp:755] [c10d] The IPv6 network addresses of (h0a1c8b098ed-node-0-0.h0a1c8b098ed, 29500) cannot be retrieved (gai error: -3 - Temporary failure in name resolution).
[W114 15:5